## 1. Environment Setup

In [ ]:
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)

In [ ]:
# Install Flash Attention (optional but recommended for A100/H100)
# !pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.5.4/flash_attn-2.6.3+cu124torch2.9-cp312-cp312-linux_x86_64.whl

In [ ]:
# Check GPU
!nvidia-smi

# Check if we're on Colab
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"Running on Colab: {IN_COLAB}")

In [ ]:
# Clone repository (if on Colab)
import os

REPO_URL = "https://github.com/Pkansagra-hub/Family_osModernBERT.git"
REPO_DIR = "Modeling_studio"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("Repository already exists, pulling latest...")
        !cd {REPO_DIR} && git pull

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")
else:
    print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -q -e .
!pip install -q wandb tensorboard datasets>=2.14.0 accelerate
print("Dependencies installed!")

In [ ]:
# Mount Google Drive (for persistent storage)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create directories on Drive
    DRIVE_BASE = "/content/drive/MyDrive/FamilyOS_ModernBERT_v3"
    !mkdir -p "{DRIVE_BASE}/outputs"
    !mkdir -p "{DRIVE_BASE}/checkpoints"

    # Symlink outputs to Drive for persistence
    !rm -rf outputs 2>/dev/null
    !ln -s "{DRIVE_BASE}/outputs" outputs

    print(f"Outputs will be saved to: {DRIVE_BASE}")

In [ ]:
# Suppress TensorFlow warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## 2. Check Data Availability

In [ ]:
import os
from pathlib import Path

def check_v3_data():
    """Verify required data directories for v3 training."""

    # Phase 0.5 - Healing data
    healing_paths = {
        "Healing - Civil Comments": "data/public/civil_comments_curated",
        "Healing - SST2": "data/healing/sst2",
    }

    # Phase 1 - Multi-task data
    phase1_paths = {
        "Phase 1 - Emotions": "data/familyos/emotions/silver",
        "Phase 1 - Temporal": "data/familyos/temporal/silver",
        "Phase 1 - Unified": "data/familyos/unified/output",
    }

    # v3 initialized model
    v3_model_paths = {
        "v3 Initialized Model": "checkpoints/modernbert-v3-initialized",
    }

    print("Checking Phase 0.5 (Healing) data...")
    healing_ok = True
    for name, path in healing_paths.items():
        exists = os.path.exists(path)
        status = "OK" if exists else "MISSING"
        print(f"   [{status}] {name}: {path}")
        if not exists:
            healing_ok = False

    print("\nChecking Phase 1 (Multi-Task) data...")
    phase1_ok = True
    for name, path in phase1_paths.items():
        exists = os.path.exists(path)
        status = "OK" if exists else "MISSING"
        print(f"   [{status}] {name}: {path}")
        if not exists:
            phase1_ok = False

    print("\nChecking v3 Model...")
    v3_ok = True
    for name, path in v3_model_paths.items():
        exists = os.path.exists(path)
        status = "OK" if exists else "MISSING"
        print(f"   [{status}] {name}: {path}")
        if not exists:
            v3_ok = False

    return healing_ok, phase1_ok, v3_ok

healing_ready, phase1_ready, v3_ready = check_v3_data()

if not v3_ready:
    print("\n[WARNING] v3 model not initialized! Run initialization first.")
elif not healing_ready:
    print("\n[WARNING] Healing data missing - Phase 0.5 may fail.")
elif not phase1_ready:
    print("\n[WARNING] Phase 1 data missing.")
else:
    print("\n[OK] All data ready for v3 training!")

## 3. Configuration

In [ ]:
# Training Configuration
import os

# Output directory
OUTPUT_DIR = "outputs/v3_full"

# Debug mode: Set to True for quick testing (5 steps per phase)
DEBUG_RUN = False

# Which phases to run
RUN_PHASE_0_5 = True   # Enhanced Healing
RUN_PHASE_1 = True     # Multi-Task FamilyOS
RUN_PHASE_1_5 = True   # Forgetting Evaluation
RUN_PHASE_2 = True     # Fine-Tuning

# W&B logging (disable for debug runs)
USE_WANDB = not DEBUG_RUN

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Debug mode: {DEBUG_RUN}")
print(f"W&B logging: {USE_WANDB}")
print(f"\nPhases to run:")
print(f"   Phase 0.5 (Healing):     {RUN_PHASE_0_5}")
print(f"   Phase 1 (Multi-Task):    {RUN_PHASE_1}")
print(f"   Phase 1.5 (Forgetting):  {RUN_PHASE_1_5}")
print(f"   Phase 2 (Fine-Tuning):   {RUN_PHASE_2}")

In [ ]:
# Pull latest code before training
!git pull origin main

---
## 4. Phase 0.5: Enhanced Healing

**Purpose:** Preserve base model knowledge while adapting to new architecture.

- Trains on healing datasets (Civil Comments, SST-2, etc.)
- Uses knowledge distillation from base ModernBERT
- 2,500 steps (or 5 in debug mode)

In [ ]:
%%time

import os

if RUN_PHASE_0_5:
    print("="*60)
    print("PHASE 0.5: Enhanced Healing")
    print("="*60)

    # Build command
    cmd = f"python -u scripts/v3_scripts/train_v3_phase0_5.py "
    cmd += f"--config configs/training/multitask/stage_v3_phase0_5_enhanced.yaml "
    cmd += f"--output-dir {OUTPUT_DIR}/phase_0.5 "

    if DEBUG_RUN:
        cmd += "--max-steps 5 --debug "

    if not USE_WANDB:
        cmd += "--no-wandb "
    else:
        cmd += "--wandb-run-name v3_phase_0.5 "

    print(f"Command: {cmd}\n")
    !{cmd}

    # Check success
    phase_0_5_ok = os.path.exists(f"{OUTPUT_DIR}/phase_0.5/final_model/pytorch_model.bin") or \
                   os.path.exists(f"{OUTPUT_DIR}/phase_0.5/best_model/pytorch_model.bin")

    if phase_0_5_ok:
        print("\n" + "="*60)
        print("PHASE 0.5 COMPLETED SUCCESSFULLY!")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("PHASE 0.5 FAILED! Check logs above.")
        print("="*60)
else:
    print("Phase 0.5 skipped (RUN_PHASE_0_5 = False)")
    phase_0_5_ok = os.path.exists(f"{OUTPUT_DIR}/phase_0.5/best_model")

In [ ]:
# Verify Phase 0.5 output
import os
import json

phase_0_5_output = f"{OUTPUT_DIR}/phase_0.5"

if os.path.exists(phase_0_5_output):
    print(f"Phase 0.5 output: {phase_0_5_output}")

    # Check for model
    for model_dir in ["best_model", "final_model"]:
        model_path = os.path.join(phase_0_5_output, model_dir)
        if os.path.exists(model_path):
            print(f"   Model: {model_dir}/")
            for f in os.listdir(model_path):
                size = os.path.getsize(os.path.join(model_path, f)) / 1e6
                print(f"      {f} ({size:.1f} MB)")

    # Load results
    results_path = os.path.join(phase_0_5_output, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        print("\nPhase 0.5 Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"Phase 0.5 output not found at {phase_0_5_output}")

---
## 5. Phase 1: Multi-Task FamilyOS Training

**Purpose:** Main training on FamilyOS multi-task data.

- NER, Sentiment, Emotions, Intent, Temporal parsing
- Uses model from Phase 0.5
- 10,000 steps (or 5 in debug mode)

In [ ]:
%%time

import os

if RUN_PHASE_1:
    print("="*60)
    print("PHASE 1: Multi-Task FamilyOS")
    print("="*60)

    # Get model from Phase 0.5
    model_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_0.5/{model_dir}"
        if os.path.exists(candidate):
            model_path = candidate
            break

    if not model_path:
        print("ERROR: No model from Phase 0.5 found!")
        phase_1_ok = False
    else:
        print(f"Using model: {model_path}")

        # Build command
        cmd = f"python -u scripts/v3_scripts/train_v3_phase1.py "
        cmd += f"--config configs/training/multitask/stage_v3_phase1.yaml "
        cmd += f"--output-dir {OUTPUT_DIR}/phase_1 "
        cmd += f"--model-path {model_path} "

        if DEBUG_RUN:
            cmd += "--max-steps 5 --debug "

        if not USE_WANDB:
            cmd += "--no-wandb "
        else:
            cmd += "--wandb-run-name v3_phase_1 "

        print(f"Command: {cmd}\n")
        !{cmd}

        # Check success
        phase_1_ok = os.path.exists(f"{OUTPUT_DIR}/phase_1/final_model/pytorch_model.bin") or \
                     os.path.exists(f"{OUTPUT_DIR}/phase_1/best_model/pytorch_model.bin")

        if phase_1_ok:
            print("\n" + "="*60)
            print("PHASE 1 COMPLETED SUCCESSFULLY!")
            print("="*60)
        else:
            print("\n" + "="*60)
            print("PHASE 1 FAILED! Check logs above.")
            print("="*60)
else:
    print("Phase 1 skipped (RUN_PHASE_1 = False)")
    phase_1_ok = os.path.exists(f"{OUTPUT_DIR}/phase_1/best_model")

In [ ]:
# Verify Phase 1 output
import os
import json

phase_1_output = f"{OUTPUT_DIR}/phase_1"

if os.path.exists(phase_1_output):
    print(f"Phase 1 output: {phase_1_output}")

    # Check for model
    for model_dir in ["best_model", "final_model"]:
        model_path = os.path.join(phase_1_output, model_dir)
        if os.path.exists(model_path):
            print(f"   Model: {model_dir}/")
            for f in os.listdir(model_path):
                size = os.path.getsize(os.path.join(model_path, f)) / 1e6
                print(f"      {f} ({size:.1f} MB)")

    # Load results
    results_path = os.path.join(phase_1_output, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        print("\nPhase 1 Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"Phase 1 output not found at {phase_1_output}")

---
## 6. Phase 1.5: Forgetting Gate Evaluation

**Purpose:** Verify the model hasn't catastrophically forgotten base knowledge.

- Evaluates on SST-2, Civil Comments, etc.
- Compares to Phase 0.5 baseline
- Must pass <2% accuracy drop threshold

In [ ]:
%%time

import os

if RUN_PHASE_1_5:
    print("="*60)
    print("PHASE 1.5: Forgetting Gate Evaluation")
    print("="*60)

    # Get model from Phase 1
    model_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_1/{model_dir}"
        if os.path.exists(candidate):
            model_path = candidate
            break

    # Get baseline from Phase 0.5
    baseline_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_0.5/{model_dir}"
        if os.path.exists(candidate):
            baseline_path = candidate
            break

    if not model_path:
        print("ERROR: No model from Phase 1 found!")
        phase_1_5_ok = False
    else:
        print(f"Evaluating model: {model_path}")
        if baseline_path:
            print(f"Baseline model: {baseline_path}")

        # Build command
        cmd = f"python -u scripts/v3_scripts/evaluate_forgetting.py "
        cmd += f"--config configs/evaluation/forgetting_gate.yaml "
        cmd += f"--output-dir {OUTPUT_DIR}/phase_1.5 "
        cmd += f"--model-path {model_path} "

        if baseline_path:
            cmd += f"--baseline {baseline_path} "

        if not USE_WANDB:
            cmd += "--no-wandb "

        print(f"Command: {cmd}\n")
        !{cmd}

        # Check results
        results_path = f"{OUTPUT_DIR}/phase_1.5/results.json"
        phase_1_5_ok = os.path.exists(results_path)

        if phase_1_5_ok:
            print("\n" + "="*60)
            print("PHASE 1.5 COMPLETED!")
            print("="*60)
        else:
            print("\n" + "="*60)
            print("PHASE 1.5 FAILED! Check logs above.")
            print("="*60)
else:
    print("Phase 1.5 skipped (RUN_PHASE_1_5 = False)")
    phase_1_5_ok = True

In [ ]:
# Check Forgetting Gate Results
import os
import json

phase_1_5_output = f"{OUTPUT_DIR}/phase_1.5"
results_path = os.path.join(phase_1_5_output, "results.json")

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

    print("="*60)
    print("FORGETTING GATE RESULTS")
    print("="*60)

    # Check if gate passed
    gate_passed = results.get("gate_passed", True)

    if "forgetting_metrics" in results:
        print("\nForgetting by Task (max allowed: 2%):")
        for task, drop in results["forgetting_metrics"].items():
            status = "PASS" if drop <= 0.02 else "FAIL"
            print(f"   [{status}] {task}: {drop:.2%}")

    if gate_passed:
        print("\n" + "="*60)
        print("FORGETTING GATE PASSED! Proceeding to Phase 2.")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("FORGETTING GATE FAILED!")
        print("Consider: Increase replay ratio and re-run Phase 1")
        print("="*60)
else:
    print(f"No forgetting results found at {results_path}")
    print("Assuming gate passed (no evaluation run).")

---
## 7. Phase 2: Fine-Tuning

**Purpose:** Final refinement and head optimization.

- Lower learning rate
- Focus on task-specific heads
- 5,000 steps (or 5 in debug mode)

In [ ]:
%%time

import os

if RUN_PHASE_2:
    print("="*60)
    print("PHASE 2: Fine-Tuning")
    print("="*60)

    # Get model from Phase 1 (not 1.5, which is evaluation only)
    model_path = None
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/phase_1/{model_dir}"
        if os.path.exists(candidate):
            model_path = candidate
            break

    if not model_path:
        print("ERROR: No model from Phase 1 found!")
        phase_2_ok = False
    else:
        print(f"Using model: {model_path}")

        # Build command
        cmd = f"python -u scripts/v3_scripts/train_v3_phase2.py "
        cmd += f"--config configs/training/multitask/stage_v3_phase2.yaml "
        cmd += f"--output-dir {OUTPUT_DIR}/phase_2 "
        cmd += f"--model-path {model_path} "

        if DEBUG_RUN:
            cmd += "--max-steps 5 --debug "

        if not USE_WANDB:
            cmd += "--no-wandb "
        else:
            cmd += "--wandb-run-name v3_phase_2 "

        print(f"Command: {cmd}\n")
        !{cmd}

        # Check success
        phase_2_ok = os.path.exists(f"{OUTPUT_DIR}/phase_2/final_model/pytorch_model.bin") or \
                     os.path.exists(f"{OUTPUT_DIR}/phase_2/best_model/pytorch_model.bin")

        if phase_2_ok:
            print("\n" + "="*60)
            print("PHASE 2 COMPLETED SUCCESSFULLY!")
            print("="*60)
        else:
            print("\n" + "="*60)
            print("PHASE 2 FAILED! Check logs above.")
            print("="*60)
else:
    print("Phase 2 skipped (RUN_PHASE_2 = False)")
    phase_2_ok = os.path.exists(f"{OUTPUT_DIR}/phase_2/best_model")

In [ ]:
# Verify Phase 2 output
import os
import json

phase_2_output = f"{OUTPUT_DIR}/phase_2"

if os.path.exists(phase_2_output):
    print(f"Phase 2 output: {phase_2_output}")

    # Check for model
    for model_dir in ["best_model", "final_model"]:
        model_path = os.path.join(phase_2_output, model_dir)
        if os.path.exists(model_path):
            print(f"   Model: {model_dir}/")
            for f in os.listdir(model_path):
                size = os.path.getsize(os.path.join(model_path, f)) / 1e6
                print(f"      {f} ({size:.1f} MB)")

    # Load results
    results_path = os.path.join(phase_2_output, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        print("\nPhase 2 Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"Phase 2 output not found at {phase_2_output}")

---
## 8. Training Summary

In [ ]:
import os
import json
from datetime import datetime

print("="*60)
print("MODERNBERT V3 TRAINING SUMMARY")
print("="*60)
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check all phases
phases = {
    "Phase 0.5 (Healing)": f"{OUTPUT_DIR}/phase_0.5",
    "Phase 1 (Multi-Task)": f"{OUTPUT_DIR}/phase_1",
    "Phase 1.5 (Forgetting)": f"{OUTPUT_DIR}/phase_1.5",
    "Phase 2 (Fine-Tuning)": f"{OUTPUT_DIR}/phase_2",
}

print("\nPhase Status:")
for name, path in phases.items():
    # Check for model or results
    has_model = os.path.exists(f"{path}/best_model") or os.path.exists(f"{path}/final_model")
    has_results = os.path.exists(f"{path}/results.json")

    if has_model or has_results:
        status = "Complete"
    else:
        status = "Not Run"

    print(f"   {name}: {status}")

# Find final model
final_model = None
for phase in ["phase_2", "phase_1", "phase_0.5"]:
    for model_dir in ["best_model", "final_model"]:
        candidate = f"{OUTPUT_DIR}/{phase}/{model_dir}"
        if os.path.exists(candidate):
            final_model = candidate
            break
    if final_model:
        break

print("\n" + "="*60)
print("FINAL MODEL")
print("="*60)
if final_model:
    print(f"   {final_model}")

    # Count parameters
    model_file = os.path.join(final_model, "pytorch_model.bin")
    if os.path.exists(model_file):
        size_mb = os.path.getsize(model_file) / 1e6
        print(f"   Size: {size_mb:.1f} MB")
else:
    print("   No model found!")

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print("   1. Backup model to Google Drive")
print("   2. Run final evaluation on held-out test set")
print("   3. Export to ONNX for inference")
print("   4. Deploy to production")
print("="*60)

---
## 9. Backup to Google Drive

In [ ]:
# Backup outputs to Google Drive
if IN_COLAB:
    import shutil
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    backup_dir = f"/content/drive/MyDrive/FamilyOS_ModernBERT_v3/runs/{timestamp}"

    print(f"Creating backup at: {backup_dir}")
    os.makedirs(backup_dir, exist_ok=True)

    # Copy each phase
    for phase in ["phase_0.5", "phase_1", "phase_1.5", "phase_2"]:
        phase_path = f"{OUTPUT_DIR}/{phase}"
        if os.path.exists(phase_path):
            shutil.copytree(
                phase_path,
                f"{backup_dir}/{phase}",
                dirs_exist_ok=True
            )
            print(f"   Backed up: {phase}")

    print(f"\nBackup complete: {backup_dir}")
else:
    print("Not on Colab - skipping Drive backup.")
    print(f"Outputs are in: {OUTPUT_DIR}")

In [ ]:
# Empty cell for notes